# Q4 — Machine Translation (Multi30k EN→DE)

**Canonical comparison:** Seq2Seq + Attention (trained from scratch) vs Helsinki-NLP Opus-MT (pretrained reference)
- Seq2Seq: 2000 train / 100 validation / 100 test
- Transformer: 100 train / 100 validation / 100 test

**Report refresh chain:** two canonical runs -> `scripts/q4_report_summary.py` -> `scripts/report_comparison_figures.py`

**Runtime:** T4 GPU recommended

> This notebook reproduces the report-facing Q4 comparison artifacts.
> Uncapped or larger-budget Multi30k runs should be treated as exploratory rather than direct replacements for the current report comparison.

## 1. Environment Setup

In [ ]:
import os, sys, json, glob, shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    print('Not running on Colab — skipping Drive mount.')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q4' if ON_COLAB else None
if DRIVE_OUTPUT:
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
if ON_COLAB:
    REPO_DIR = '/content/467-takehome'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull --ff-only
else:
    REPO_DIR = os.getcwd()
    if not os.path.exists(os.path.join(REPO_DIR, 'configs')):
        REPO_DIR = os.path.abspath('..')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(
        f'  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB'
    )

In [ ]:
import gc
import re
import shlex
import subprocess
from pathlib import Path

PYTHON = sys.executable
REPO_ROOT = Path.cwd()
Q4_REFERENCE_ARTIFACTS = {
    'transformer': 'outputs/q4/run_20260413_212828',
    'seq2seq': 'outputs/q4/run_20260413_214229',
    'summary': 'outputs/q4/run_20260413_231508',
}
Q4_RUN_DIRS = {}
Q4_SUMMARY_DIR = None


def run_logged_command(cmd):
    print('$ ' + shlex.join(cmd))
    output_lines = []
    with subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            output_lines.append(line)
        return_code = process.wait()
    output = ''.join(output_lines)
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd, output=output)
    return output


def extract_saved_path(command_output, prefix):
    match = re.search(rf'{re.escape(prefix)} (.+)', command_output)
    if not match:
        raise RuntimeError(f"Could not find '{prefix}' in command output.")
    return Path(match.group(1).strip()).resolve()


def run_q4_model(model_key, extra_overrides=None):
    cmd = [
        PYTHON,
        '-m',
        'src.q4_machine_translation.main',
        '--config',
        'configs/q4.yaml',
        '--final-eval',
        '--override',
        *(extra_overrides or []),
    ]
    output = run_logged_command(cmd)
    run_dir = extract_saved_path(output, 'Saved outputs to')
    Q4_RUN_DIRS[model_key] = run_dir
    print(f'Registered {model_key} run: {run_dir}')
    return run_dir


def copy_artifact_to_drive(path, label):
    artifact_path = Path(path)
    if not DRIVE_OUTPUT:
        print(f'Drive output not configured; keeping artifact locally at {artifact_path}')
        return artifact_path
    if artifact_path.is_dir():
        destination = Path(DRIVE_OUTPUT) / f'{artifact_path.name}_{label}'
        shutil.copytree(artifact_path, destination, dirs_exist_ok=True)
    else:
        destination = Path(DRIVE_OUTPUT) / f'{artifact_path.stem}_{label}{artifact_path.suffix}'
        shutil.copy2(artifact_path, destination)
    print(f'Copied {artifact_path} -> {destination}')
    return destination


def build_q4_summary():
    global Q4_SUMMARY_DIR
    missing_models = {'seq2seq', 'transformer'} - set(Q4_RUN_DIRS)
    if missing_models:
        raise RuntimeError(f'Run these models first: {sorted(missing_models)}')
    cmd = [
        PYTHON,
        'scripts/q4_report_summary.py',
        '--run',
        str(Q4_RUN_DIRS['seq2seq']),
        '--run',
        str(Q4_RUN_DIRS['transformer']),
    ]
    output = run_logged_command(cmd)
    Q4_SUMMARY_DIR = extract_saved_path(output, 'Saved Q4 report summary outputs to')
    print(f'Registered Q4 summary artifact: {Q4_SUMMARY_DIR}')
    return Q4_SUMMARY_DIR


def refresh_q4_report_figure(summary_dir=None):
    if summary_dir is None and Q4_SUMMARY_DIR is None:
        raise RuntimeError('Build the Q4 summary artifact before refreshing the report figure.')
    summary_root = Path(summary_dir or Q4_SUMMARY_DIR)
    cmd = [
        PYTHON,
        'scripts/report_comparison_figures.py',
        '--q4-summary',
        str(summary_root / 'q4_report_summary.json'),
    ]
    run_logged_command(cmd)
    figure_path = REPO_ROOT / 'report' / 'figures' / 'q4' / 'test_metric_comparison.png'
    print(f'Refreshed Q4 report figure: {figure_path}')
    return figure_path


def gpu_cleanup():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    allocated_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    print(f'GPU memory freed. Allocated: {allocated_gb:.2f} GB')


print(json.dumps(Q4_REFERENCE_ARTIFACTS, indent=2))

## 2. Seq2Seq+Attention (GPU — Training)

Trained from scratch on 2000 train samples. Uses GRU encoder-decoder with attention.

In [ ]:
seq2seq_run = run_q4_model(
    'seq2seq',
    extra_overrides=[
        'models.seq2seq.enabled=true',
        'models.transformer.enabled=false',
        'dataset.limit_train_samples=2000',
        'dataset.limit_validation_samples=100',
        'dataset.limit_test_samples=100',
        'preprocess.lowercase_source=true',
        'preprocess.lowercase_target=true',
        'models.seq2seq.embedding_dim=128',
        'models.seq2seq.hidden_dim=128',
        'models.seq2seq.num_layers=1',
        'models.seq2seq.dropout=0.2',
        'models.seq2seq.batch_size=32',
        'models.seq2seq.learning_rate=0.001',
        'models.seq2seq.weight_decay=0.0',
        'models.seq2seq.max_epochs=8',
        'models.seq2seq.early_stopping_patience=2',
        'models.seq2seq.teacher_forcing_ratio=0.5',
        'models.seq2seq.max_vocab_size=8000',
        'models.seq2seq.min_token_frequency=1',
        'models.seq2seq.max_output_length=32',
        'models.seq2seq.gradient_clip=1.0',
        'models.seq2seq.num_workers=0',
    ],
)

In [ ]:
seq2seq_drive_copy = copy_artifact_to_drive(seq2seq_run, 'seq2seq_canonical')
gpu_cleanup()

## 3. Run Canonical Transformer Reference (GPU)

This cell reproduces the pretrained Opus-MT reference artifact used in the current Q4 report comparison.

In [ ]:
transformer_run = run_q4_model(
    'transformer',
    extra_overrides=[
        'models.seq2seq.enabled=false',
        'models.transformer.enabled=true',
        'dataset.limit_train_samples=100',
        'dataset.limit_validation_samples=100',
        'dataset.limit_test_samples=100',
        'preprocess.lowercase_source=false',
        'preprocess.lowercase_target=false',
        'models.transformer.model_name=Helsinki-NLP/opus-mt-en-de',
        'models.transformer.batch_size=8',
        'models.transformer.max_input_length=96',
        'models.transformer.max_output_length=96',
        'models.transformer.num_beams=4',
        'models.transformer.length_penalty=1.0',
        'models.transformer.early_stopping=true',
    ],
)

In [ ]:
transformer_drive_copy = copy_artifact_to_drive(transformer_run, 'transformer_canonical')
gpu_cleanup()

## 4. Build the Report-Ready Summary Chain

After the two canonical runs finish, build a fresh Q4 comparison summary and refresh the report-local Q4 figure from that summary JSON.

Because the transformer is a pretrained reference and the seq2seq model trains from scratch, keep larger-budget reruns or fine-tuning work on a separate exploratory branch unless the report comparison itself is being intentionally replaced.

In [ ]:
q4_summary_dir = build_q4_summary()
q4_summary_drive_copy = copy_artifact_to_drive(q4_summary_dir, 'report_summary')
q4_figure_path = refresh_q4_report_figure(q4_summary_dir)
q4_figure_drive_copy = copy_artifact_to_drive(q4_figure_path, 'report_figure')

print('\n=== Current notebook outputs ===')
print(
    json.dumps(
        {
            'runs': {model: str(path) for model, path in Q4_RUN_DIRS.items()},
            'summary': str(q4_summary_dir),
            'report_figure': str(q4_figure_path),
            'drive_copies': {
                'seq2seq': str(seq2seq_drive_copy),
                'transformer': str(transformer_drive_copy),
                'summary': str(q4_summary_drive_copy),
                'report_figure': str(q4_figure_drive_copy),
            },
        },
        indent=2,
    )
)

print('\nReference artifacts used by the current report:')
print(json.dumps(Q4_REFERENCE_ARTIFACTS, indent=2))

## 5. Inspect the Current Metrics Snapshot

In [ ]:
print('=== Q4 Canonical Results ===')
for label, run_dir in [('Seq2Seq', seq2seq_run), ('Transformer', transformer_run)]:
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            metrics = json.load(f)
        print(f'\n--- {label} ({os.path.basename(run_dir)}) ---')
        print(json.dumps(metrics, indent=2, default=str)[:2000])
    else:
        print(f'\n--- {label}: metrics.json not found ---')

if Q4_SUMMARY_DIR is not None:
    summary_file = os.path.join(Q4_SUMMARY_DIR, 'q4_report_summary.json')
    if os.path.exists(summary_file):
        with open(summary_file) as f:
            summary_payload = json.load(f)
        print(f'\n--- Summary artifact ({os.path.basename(Q4_SUMMARY_DIR)}) ---')
        print(json.dumps(summary_payload, indent=2, default=str)[:2000])

---
## Exploratory: Larger-Budget Multi30k Runs

These commented cells keep the older uncapped direction available for follow-up experiments, but they are not part of the canonical report comparison.

If you run them, export a separate summary artifact instead of overwriting the capped transformer-versus-seq2seq chain above.

In [ ]:
# # --- Seq2Seq full Multi30k, bigger model ---
# !python -m src.q4_machine_translation.main \
#     --config configs/q4.yaml \
#     --final-eval \
#     --override \
#         models.seq2seq.enabled=true \
#         models.transformer.enabled=false \
#         models.seq2seq.max_epochs=20 \
#         models.seq2seq.early_stopping_patience=5 \
#         models.seq2seq.hidden_dim=256 \
#         models.seq2seq.embedding_dim=256 \
#         models.seq2seq.num_layers=2 \
#         models.seq2seq.dropout=0.3 \
#         models.seq2seq.batch_size=64 \
#         dataset.limit_train_samples=null \
#         dataset.limit_validation_samples=null \
#         dataset.limit_test_samples=null

# # --- Transformer full Multi30k ---
# !python -m src.q4_machine_translation.main \
#     --config configs/q4.yaml \
#     --final-eval \
#     --override \
#         models.transformer.enabled=true \
#         models.seq2seq.enabled=false \
#         models.transformer.batch_size=16 \
#         dataset.limit_train_samples=null \
#         dataset.limit_validation_samples=null \
#         dataset.limit_test_samples=null